# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import numpy as np
import pandas as pd
from itertools import product
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
import yaml
import duckdb
from datetime import datetime, timedelta

import sys
sys.path.append(str(Path(globals()['_dh'][0]).resolve().parent.parent))

from generator.paths import project_path, pipeline_path, input_path, data_path, output_path
from generator.library.utilities import sort_params, clear_dir, ktok
from generator.library.loaders import base_loader_csv
from generator.transformers.dimension import segment_constant_factor_split, segment_add_total, segment_append_constant, segment_append_percentage, segment_append_reminder, segment_remove, timestamp_append_extend
from generator.library.db import read_data, delete_duckdb_file
import generator.library.scenario_constraints


In [ ]:
# 2. Load configuration (new version, come back to this later)

with open(project_path / 'config.yaml', "r") as f:
    config = yaml.safe_load(f)

with open(pipeline_path / 'county-prototype.yaml', "r") as f:
    pipeline = yaml.safe_load(f)

## Set flag to clear the api folder
clear_api_flag = True

## Set common variables

data_path = output_path
db_file = data_path / "core.duckdb"
default_table = "demand"

base_partition = ['geography', 'segment', 'timestamp_year']
full_partition = ['scenario_id', 'geography', 'segment', 'timestamp_year']

base_schema = ['geography', 'segment', 'timestamp', 'value']

base_schema_map = {
    'geography': "geography",
    'segment': "segment", 
    'timestamp': "timestamp", 
    'value': "value"
}

scenario_schema = ['scenario_id']

full_schema = scenario_schema + base_schema

'''
# This is the full set of scenarios

scenario_schema = [
    'housing_electricification',
    'transport_electrification',
    'industry_transition',
    'population',
    'flexibility',
    'new_industry',
    'new_datacenters',
    'energy_efficiency'
    ]
'''

base_data = "base-demand/base-load-curve,aggregation=mean,base-year=2024,geography=00,normalized=False,resolution=1h.csv"

In [3]:
# Clear the output path to prepare for new data

clear_dir(output_path)

In [4]:
# 3. Calculate the scenarios

# Extract names and values
names = [obj['name'] for obj in config['scenario']['scenarios']]
values = [[item['value'] for item in obj['items']] for obj in config['scenario']['scenarios']]

# Build default scenario
default_scenario = {
    obj["name"]: obj["default"]
    for obj in config['scenario']['scenarios']
    if "default" in obj
}

# Generate all scenarios
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Filter using constraints and mark the default
scenarios = []
for s in all_scenarios:
    s_out = dict(s)
    if s == default_scenario:
        s_out["default"] = True
    scenarios.append(s_out)

In [5]:
# 3. Load the base demand

base_loader_csv(base_schema, base_schema_map, input_path / base_data, db_file)

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'demand',
 'added_rows': 8760,
 'added_columns': ['geography', 'segment', 'timestamp', 'value']}

In [ ]:
# 4 Segment the data (temporary)

# 4.1 Split the data by geographies
geo_file = input_path / "county_energy_split/county_energy_split.csv"
geographies = pd.read_csv(geo_file, dtype={"geography": str, "factor": float})

segment_constant_factor_split(db_file, geographies)

# 4.2 Split the data by segments
segment_append_constant(db_file, 'segment', 'industry', 100)
segment_append_percentage(db_file, 'segment', 'total', 'transport', 0.15)
segment_append_reminder(db_file, 'segment', 'total', ['industry', 'transport'], 'housing')
segment_remove(db_file, 'segment', 'total')

,geography,segment,timestamp,value
0,23,transport,2024-06-20 16:00:00,24.497446
1,23,transport,2024-06-20 17:00:00,23.933727
2,23,transport,2024-06-20 18:00:00,23.518705
3,23,transport,2024-06-20 19:00:00,22.975017
4,23,transport,2024-06-20 20:00:00,22.404685
...,...,...,...,...
551875,22,housing,2024-09-13 19:00:00,707.547879
551876,22,housing,2024-09-13 20:00:00,682.689863
551877,22,housing,2024-09-13 21:00:00,655.493522
551878,22,housing,2024-09-13 22:00:00,634.537482


In [8]:
timestamp_append_extend(
    in_data=db_file, 
    base_year=2024,
    start_year=2025,
    end_year=2050,
    out_data=data_path, 
    partition=base_partition
)

delete_duckdb_file(db_file)

True

In [12]:
read_data(data_path)

,geography,segment,timestamp,value
0,01,housing,2025-01-01 00:00:00,1446.157488
1,01,housing,2025-01-01 01:00:00,1426.971654
2,01,housing,2025-01-01 02:00:00,1419.869248
3,01,housing,2025-01-01 03:00:00,1421.536417
4,01,housing,2025-01-01 04:00:00,1443.718037
...,...,...,...,...
14357947,25,transport,2050-12-31 19:00:00,201.031862
14357948,25,transport,2050-12-31 20:00:00,197.737037
14357949,25,transport,2050-12-31 21:00:00,194.098392
14357950,25,transport,2050-12-31 22:00:00,189.187429


**The code below all pertains to writing data.**

In [ ]:
## Clear the api folder
if clear_api_flag:
    print("Clearing API folder...")
    clear_dir(data_path)

Clearing API folder...


In [ ]:
# Partition dataframe and write to parquet with pyarrow

use_scenario_id = config['generator']['useScenarioId']
partition_keys = config['generator']['partitionKeys']
row_group_size = config['generator']['rowGroupSize']

# 1) If scenario._id is used, compute and append it directly in the DataFrame
if use_scenario_id:
    print("Adding scenario._id to DataFrame...")

    scenario_cols = sorted(
        col for col in extended_geography_scenario_sector_demand.columns
        if col.startswith('scenario.')
    )

    extended_geography_scenario_sector_demand['scenario._id'] = (
        extended_geography_scenario_sector_demand[scenario_cols]
        .astype(str)
        .rename(columns=lambda col: col.split('.', 1)[1])  # remove "scenario." prefix
        .agg(lambda row: '+'.join(f"{k}:{v}" for k, v in row.items()), axis=1)
    )

# 2) Convert your pandas DF into an Arrow Table
print("Converting DataFrame to Arrow Table...")
table = pa.Table.from_pandas(
    extended_geography_scenario_sector_demand,
    preserve_index=False
)

actual_partitions = []

# 3) Compute derived keys and append
print("Computing derived partition keys...")
for entry in partition_keys:
    if isinstance(entry, dict) and "transform" in entry:
        src = entry["path"]
        tf  = entry["transform"]
        arr = getattr(pc, tf)(table[src])
        table = table.append_column(tf, arr)
        actual_partitions.append(tf)
    elif isinstance(entry, dict):
        actual_partitions.append(entry['path'])

# 4) Write Parquet dataset partitioned on actual columns
print("Writing Parquet dataset...")
ds.write_dataset(
    data=table,
    base_dir=str(data_path),
    format="parquet",
    partitioning=actual_partitions,
    partitioning_flavor="hive",
    file_options=ds.ParquetFileFormat()
                   .make_write_options(compression="snappy"),
    min_rows_per_group=row_group_size,
    max_rows_per_group=row_group_size,
    existing_data_behavior="overwrite_or_ignore"
)


Adding scenario._id to DataFrame...
Converting DataFrame to Arrow Table...
Computing derived partition keys...
Writing Parquet dataset...


In [ ]:
extended_geography_scenario_sector_demand

,dimensions.segment.level1,dimensions.geography,period.start,scenario.growth,value,scenario._id
0,industry,01,2025-01-01 00:00:00+00:00,0,554.651596,growth:0
1,industry,01,2025-01-01 01:00:00+00:00,0,554.651596,growth:0
2,industry,01,2025-01-01 02:00:00+00:00,0,554.651596,growth:0
3,industry,01,2025-01-01 03:00:00+00:00,0,554.651596,growth:0
4,industry,01,2025-01-01 04:00:00+00:00,0,554.651596,growth:0
...,...,...,...,...,...,...
33135475,transport,25,2044-12-31 19:00:00+00:00,2,147.583130,growth:2
33135476,transport,25,2044-12-31 20:00:00+00:00,2,149.283653,growth:2
33135477,transport,25,2044-12-31 21:00:00+00:00,2,136.600535,growth:2
33135478,transport,25,2044-12-31 22:00:00+00:00,2,140.067808,growth:2


In [ ]:
# Partition dataframe and write to parquet with pyarrow (OLD)
'''
use_scenario_id = config['generator']['useScenarioId']
partition_keys = config['generator']['partitionKeys']
row_group_size = config['generator']['rowGroupSize']

# 1) Convert your pandas DF into an Arrow Table
table = pa.Table.from_pandas(
    extended_geography_scenario_sector_demand,
    preserve_index=False
)

# TODO: Rewrite a more elegant function for transforming the partitioning.yaml (maybe change the format to a schema)
# 2) Compute each derived key in Arrow and append as a real column
#    We’ll also build a list of the *actual* column names to partition on:
actual_partitions = []
for entry in partition_keys:
    if isinstance(entry, dict) and "transform" in entry:
        src = entry["path"]            # e.g. "period.start"
        tf  = entry["transform"]       # e.g. "year"
        # Compute via pyarrow.compute.<tf>(...)
        arr = getattr(pc, tf)(table[src])
        # Append that as a column named exactly tf
        table = table.append_column(tf, arr)
        actual_partitions.append(tf)
    else:
        # plain existing column
        actual_partitions.append(entry['path'])

# 3) Now write a Hive‐style dataset over those *real* columns
ds.write_dataset(
    data=table,
    base_dir=str(data_path),
    format="parquet",
    partitioning=actual_partitions,         # list[str] works with hive flavor
    partitioning_flavor="hive",
    file_options=ds.ParquetFileFormat()
                   .make_write_options(compression="snappy"),
    min_rows_per_group=row_group_size,
    max_rows_per_group=row_group_size,
    existing_data_behavior="overwrite_or_ignore"
)
'''